<center><img src="../../image/dice.png" alt="DICE" style="width:300px;"/></center>

# 9. 四方块自动堆叠

@－－－－DICE－－－－<br>
@ 文件名：09.四方块自动堆叠.ipynb <br>
@ 版本：V1.0 <br>
@ 说明：从 A/B/C/D 四个抓取区依次取块，在中央堆叠区自动堆成 4 层塔，并完成连续 2 轮稳定性测试 <br>

> 本 Notebook 复用 `2.机械臂实验` 中的 `Arm5DoFUServo`、`Arm5DoFKinematic`、`icp_solver_svd`、`Transform`，
> 以及 `08.四个堆叠方块归位.ipynb` 的 Pick & Place 框架：把第三章“中央抓取 → 四区放置”的归位逻辑反向，
> 改造成“四区抓取 → 中央逐层堆叠”，层号 `stack_level` 作为唯一变化的参数。
> 九点标定数据采用 2026-10 的新九点实测值，不再沿用旧工作台参数。

## 1. 运行模式与依赖

- `RUN_MODE = "sim"`：离线运动学与流程仿真，不连接机械臂，可直接从头运行到结尾。
- `RUN_MODE = "real"`：连接真机。首次运行前必须确认工作台区域无人、急停按钮可用，并建议先只跑一轮。
- 仿真模式用于检查：九点标定、工作台坐标变换、A/B/C/D 抓取位姿、四层堆叠目标高度、逆运动学可达性、动作顺序、退出路径和塔体稳定性。
- 真机模式复用原实验的串口舵机 SDK，`gripper_open/close`、`set_tool_pose` 等均会等待舵机到位。
- 与 `08.四个堆叠方块归位.ipynb` 的差别：08 是“自上而下拆塔、四区归位”，本实验是“四区取块、自下而上码塔”，两者的 Pick & Place 六阶段流程完全一致，只交换了抓取点与放置点。

In [ ]:
%matplotlib inline
import os
import sys
import math
import time
import json
import csv
from dataclasses import dataclass, field, asdict
from datetime import datetime

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.lines import Line2D
from IPython.display import display, HTML

# ===================== 运行模式 =====================
RUN_MODE = "sim"          # "sim" 或 "real"
REAL_CONFIRM = False      # 真机运行前改为 True
REAL_ROUNDS = 1           # 真机建议先跑 1 轮

# 中文显示
plt.rcParams["font.sans-serif"] = [
    "SimHei", "Microsoft YaHei", "WenQuanYi Micro Hei",
    "Noto Sans CJK SC", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False
np.set_printoptions(precision=4, suppress=True)

RESULT_DIR = "results"
IMAGE_DIR = "image"
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
print("RUN_MODE =", RUN_MODE)
print("工作目录 =", os.getcwd())

## 2. 九点标定与工作台坐标变换

九点标定板尺寸按 `config/handeye_9points.yaml` 取 **宽 133 mm × 高 100 mm**。  
工作台坐标系原点位于 P4，X 轴指向 P1/P2 一侧，Y 轴指向 P0/P6 一侧。  
先把九点 Z 值按“半径—Z 误差”做一次线性平面补偿，再用 SVD-ICP 求 `T_arm2ws`。  

> 标定板与 `08.四个堆叠方块归位.ipynb` 使用同一组 2026-10 新九点实测数据，保证拆塔与码塔两个实验共用同一套工作台坐标系。

In [ ]:
from icp import icp_solver_svd
from transform import Transform
from arm5dof_kinematic import Arm5DoFKinematic
from config import ARM_POSE_DICT, PITCH_PANEL_A, PITCH_PANEL_B, PITCH_PANEL_C

# ===================== 2026-10 新九点实测数据（机械臂坐标，mm） =====================
ARM_9P_MEASURED = np.array([
    [236.689,  48.427,  -8.298],
    [230.298, -12.795, -14.189],
    [229.308, -77.589,  -0.554],
    [190.426,  53.205,  -3.096],
    [194.361,  -7.849,  -4.952],
    [191.712, -66.670,  -2.420],
    [150.223,  52.242,   0.724],
    [142.433,  -8.634,  -1.878],
    [139.329, -63.991,  -0.532],
], dtype=np.float64)

BOARD_WIDTH_MM = 133.0
BOARD_HEIGHT_MM = 100.0
X0 = BOARD_HEIGHT_MM / 2.0      # P0/P1/P2 的 X
Y0 = BOARD_WIDTH_MM / 2.0       # P0/P3/P6 的 Y

# 九点标定板在工作台坐标系下的理论坐标（原点 P4）
WS_9P = np.float64([
    [ X0,  Y0, 0.0],   # P0
    [ X0,  0.0, 0.0],   # P1
    [ X0, -Y0, 0.0],   # P2
    [0.0,  Y0, 0.0],   # P3
    [0.0,  0.0, 0.0],   # P4
    [0.0, -Y0, 0.0],   # P5
    [-X0,  Y0, 0.0],   # P6
    [-X0,  0.0, 0.0],   # P7
    [-X0, -Y0, 0.0],   # P8
])

# 关节臂的 Z 值与半径相关，先拟合 z_error(r)，再把九点 Z 拉平
Z_MEAN = float(np.mean(ARM_9P_MEASURED[:, 2]))
RADII = np.hypot(ARM_9P_MEASURED[:, 0], ARM_9P_MEASURED[:, 1])
Z_ERROR_POLY = np.polyfit(RADII, ARM_9P_MEASURED[:, 2] - Z_MEAN, 1)  # [k, b]
ARM_9P_FLAT = ARM_9P_MEASURED.copy()
ARM_9P_FLAT[:, 2] = Z_MEAN

# 求机械臂坐标系 -> 工作台坐标系的齐次变换
R_ARM2WS, t_ARM2WS = icp_solver_svd(ARM_9P_FLAT, WS_9P)
T_ARM2WS = np.eye(4)
T_ARM2WS[:3, :3] = R_ARM2WS
T_ARM2WS[:3, 3] = np.asarray(t_ARM2WS).reshape(-1)
T_WS2ARM = Transform.inverse(T_ARM2WS)

# 标定残差
ARM_PRED = (T_ARM2WS @ np.c_[WS_9P, np.ones(len(WS_9P))].T).T[:, :3]
RESIDUAL = ARM_PRED - ARM_9P_FLAT
RESIDUAL_NORM = np.linalg.norm(RESIDUAL, axis=1)

print("Z_MEAN = {:.4f} mm".format(Z_MEAN))
print("z_error(r) = {:.4f} * r + {:.4f}".format(*Z_ERROR_POLY))
print("T_arm2ws =")
print(np.array2string(T_ARM2WS, precision=4, suppress_small=True))
print("九点残差：max = {:.3f} mm, mean = {:.3f} mm".format(
    RESIDUAL_NORM.max(), RESIDUAL_NORM.mean()))

try:
    T_SAVED = np.loadtxt("config/T_arm2ws.txt", delimiter=",")
    print("与 config/T_arm2ws.txt 的最大绝对差异 = {:.6f}".format(
        np.abs(T_SAVED - T_ARM2WS).max()))
except Exception as exc:
    print("读取 config/T_arm2ws.txt 失败（不影响本次计算）：", exc)

# 中央堆叠区（工作台坐标 P4）在机械臂坐标系下的位置，供真机对点参考
STACK_CENTER_ARM = (T_ARM2WS @ np.array([0.0, 0.0, 0.0, 1.0]))[:3]
print("中央堆叠区 (ws 原点) 在机械臂坐标系：", np.array2string(STACK_CENTER_ARM, precision=2))

## 3. 堆叠任务参数与 stack_pose 参数表

- 方块高度：`28 mm`（来自 `config/object.yaml`，本 Notebook 设为可调参数）。
- 四个抓取区沿用 `07.机械臂抓取工作台物块测试.ipynb` 的四周点位：A(55, 80)、B(55, -80)、C(-55, -80)、D(-55, 80) mm。
- 中央堆叠区取工作台原点 P4 `(0, 0)`，抓取顺序为 **A → B → C → D**，对应自下而上的第 1～4 层。
- 层号 `stack_level` 从 0 开始计数，第 1～4 层对应的目标中心高度由 `stack_z(level)` 统一计算，不再为每层单独写死坐标。
- 安全高度 `200 mm`；放置上方点比释放点高 `60 mm`；释放点 = 目标高度 + `release_clearance`。

In [ ]:
# ===================== 堆叠任务参数 =====================
# 方块高度优先从 config/object.yaml 读取；仿真环境没有 PyYAML 时回退到 28 mm
try:
    import yaml
    with open("config/object.yaml", "r", encoding="utf-8") as _f:
        CUBE_SIZE_MM = float(yaml.safe_load(_f).get("height", 28.0))
except Exception:
    CUBE_SIZE_MM = 28.0

# A/B/C/D 四个抓取区（工作台坐标，与 07.机械臂抓取工作台物块测试.ipynb 一致）
ZONES = {
    "A": ( 55.0,  80.0),   # 右前 / P0 外侧
    "B": ( 55.0, -80.0),   # 右后 / P2 外侧
    "C": (-55.0, -80.0),   # 左后 / P8 外侧
    "D": (-55.0,  80.0),   # 左前 / P6 外侧
}
PICK_ORDER = ["A", "B", "C", "D"]      # 抓取顺序，对应自下而上第 1～4 层

STACK_XY = (0.0, 0.0)                  # 中央堆叠区中心（工作台原点 P4）
SAFE_Z_MM = 200.0                      # 安全高度：所有横移都在此高度完成
APPROACH_CLEARANCE_MM = 60.0           # 抓取/放置上方点相对目标点的抬高量
RELEASE_CLEARANCE_MM = 1.0             # 释放点比目标中心高 1 mm，避免夹爪压住方块
ZONE_PICK_Z_MM = CUBE_SIZE_MM / 2.0    # 抓取区方块中心高度 = 14 mm

OPEN_ANGLE_RAD = math.radians(30.0)

@dataclass
class StackConfig:
    """堆叠任务参数表：位姿数据与动作逻辑分离，层号 stack_level 是唯一变化的输入。"""
    cube_size: float = CUBE_SIZE_MM
    zones: dict = field(default_factory=lambda: dict(ZONES))
    pick_order: tuple = field(default_factory=lambda: tuple(PICK_ORDER))
    stack_xy: tuple = STACK_XY
    safe_z: float = SAFE_Z_MM
    approach_clearance: float = APPROACH_CLEARANCE_MM
    release_clearance: float = RELEASE_CLEARANCE_MM
    zone_pick_z: float = ZONE_PICK_Z_MM
    open_angle: float = OPEN_ANGLE_RAD
    pick_steps: int = 5
    place_steps: int = 8
    transfer_steps: int = 6
    retreat_steps: int = 6
    settle_s: float = 0.25

    # ---------- 层号 → 高度：堆叠实验的核心公式 ----------
    def stack_z(self, level: int) -> float:
        """第 level 层（0 开始计数）方块中心的目标高度。"""
        return self.cube_size * (level + 0.5) + self.release_clearance

    def tower_top_z(self, level: int) -> float:
        """放完第 level 层之后的塔顶高度（不含释放间隙）。"""
        return self.cube_size * (level + 1)

    def approach_z(self, level: int) -> float:
        """第 level 层的放置上方点高度。"""
        return self.stack_z(level) + self.approach_clearance

    def plan(self):
        """返回 [(层号, 抓取区), ...]，层号自下而上 0～3。"""
        return [(i, zone) for i, zone in enumerate(self.pick_order)]

cfg = StackConfig()

print("中央堆叠区中心：", cfg.stack_xy, " 方块高度：{:.1f} mm".format(cfg.cube_size))
print("抓取顺序：", " → ".join(cfg.pick_order), "  四层塔总高：{:.1f} mm".format(cfg.cube_size * 4))
print()
print("{:<8} {:<8} {:>10} {:>12} {:>12} {:>12} {:>10}".format(
    "层号", "抓取区", "抓取点Z", "抓取上方点Z", "目标中心Z", "放置上方点Z", "塔顶Z"))
for level, zone in cfg.plan():
    print("{:<8} {:<8} {:>10.1f} {:>12.1f} {:>12.1f} {:>12.1f} {:>10.1f}".format(
        "第{}层".format(level + 1), zone, cfg.zone_pick_z,
        cfg.zone_pick_z + cfg.approach_clearance,
        cfg.stack_z(level), cfg.approach_z(level), cfg.tower_top_z(level)))

# stack_level → 目标高度 的自动计算（本实验唯一的“位姿公式”）
print()
print("stack_level 自动高度：", [
    (lv + 1, round(cfg.stack_z(lv), 1)) for lv, _ in cfg.plan()])

## 4. 仿真机械臂（离线验证）

`VirtualArm` 不连接串口，只用 `Arm5DoFKinematic` 做逆解、正解和关节限位检查，并复现真机 SDK 的以下行为：

1. `set_tool_pose()` 先做 `z_error(r)` 补偿，再自动生成俯仰角；
2. 逆解无解时返回 `False`；
3. `gripper_open/close` 记录夹爪状态；
4. `home()` 回到 `ARM_POSE_DICT["home"]`。

因此同一套 `StackTask` 可以同时跑仿真和真机，仿真与真机只差一个 `RUN_MODE`。

In [ ]:
class VirtualArm:
    """只做运动学验证的虚拟机械臂，接口与 Arm5DoFUServo 保持兼容。"""
    is_virtual = True

    def __init__(self, T_arm2ws, z_error_polyfit):
        self.T_arm2ws = np.asarray(T_arm2ws, dtype=np.float64)
        self.z_error_polyfit = np.asarray(z_error_polyfit, dtype=np.float64)
        self.kin = Arm5DoFKinematic()
        self.joints = np.array(self.kin.JOINT_ANGLE_DEFAULT[:5], dtype=np.float64)
        self.pose = np.array(self.kin.forward_kinematic_v2(self.joints), dtype=np.float64)
        self.gripper = "open"
        self.command_count = 0
        self.ik_fail = 0

    def _adjust(self, p):
        x, y, z = p
        r = math.hypot(x, y)
        k, b = self.z_error_polyfit
        return [x, y, z + k * r + b]

    def _auto_pitch(self, p):
        x, y, z = p
        return PITCH_PANEL_A * math.hypot(x, y) + PITCH_PANEL_B * z + PITCH_PANEL_C

    def set_tool_pose(self, tool_posi=None, tool_pitch=None, tool_roll=np.pi,
                      pose_name=None, T=1.0, **kwargs):
        if pose_name is not None:
            x, y, z, tool_pitch, tool_roll = ARM_POSE_DICT[pose_name]
            tool_posi = [x, y, z]
        p = self._adjust(tool_posi)
        if tool_pitch is None:
            tool_pitch = self._auto_pitch(p)
        sols = self.kin.inverse_kinematic(
            p[0], p[1], p[2], tool_pitch, tool_roll,
            last_joint_angle=self.joints)
        if len(sols) == 0:
            self.ik_fail += 1
            return False
        sols = np.asarray(sols, dtype=np.float64)
        idx = int(np.argmin(np.linalg.norm(sols - self.joints, axis=1)))
        self.joints = sols[idx]
        self.pose = np.asarray(
            self.kin.forward_kinematic_v2(self.joints), dtype=np.float64)
        self.command_count += 1
        return True

    def home(self):
        return self.set_tool_pose(pose_name="home", T=2.0)

    def gripper_open(self, angle=np.pi / 6, T=0.5, **kwargs):
        self.gripper = "open"
        return True

    def gripper_close(self, T=0.5, **kwargs):
        self.gripper = "closed"
        return True

    def set_damping(self, power=0):
        pass

    def get_tool_pose(self):
        return self.pose.copy()

print("VirtualArm 定义完成")


## 5. 堆叠控制器（复用第 3 章 Pick & Place 逻辑）

栈式堆叠与第 3 章的归位共用同一套六阶段动作，只是把“抓取点”与“放置点”互换：

| 阶段 | 第 3 章 归位（08） | 本章 堆叠（09） |
| --- | --- | --- |
| Approach | 中央堆叠顶上方 | 抓取区上方 |
| Pick | 中央塔顶抓块 | A/B/C/D 区抓块 |
| Lift | 抬升离开塔 | 抬升离开工作台 |
| Transfer | 横移到目标区 | 横移到中央堆叠区 |
| Place | 放到目标区（Z=14 mm） | 放到第 `stack_level` 层（Z 随层号变化） |
| Retreat | 抬升退出 | **纯垂直**抬升退出，不拖动已放方块 |

函数封装：

- `pick_from(zone)`：从 A/B/C/D 抓取区取一块方块，只做抓取，不关心它要去哪一层；
- `place_at_level(level)`：把手中的方块放到第 `level` 层，目标高度由 `cfg.stack_z(level)` 自动计算；
- `move_z()` / `move_xy()`：把大幅动作拆成小步，避免关节插补产生大幅斜向扫掠；
- `run()`：按 A→B→C→D 顺序循环调用上面两个函数，四层塔一次成型，**不存在第二套重复代码**。

In [ ]:
class SafetyError(RuntimeError):
    pass

class StackTask:
    """四方块自动堆叠：A/B/C/D 四区抓取 → 中央逐层码塔。

    与 08.四个堆叠方块归位.ipynb 的 StackUnstackTask 共用同一套六阶段流程，
    区别只有两点：
      1. 抓取点来自 cfg.zones（四区），放置点是中央堆叠区的 stack_z(level)；
      2. 放置完成后只做垂直退出，绝不带着夹爪在低 Z 区横移，避免拖倒新放的方块。
    """
    def __init__(self, arm, T_arm2ws, config):
        self.arm = arm
        self.T_arm2ws = np.asarray(T_arm2ws, dtype=np.float64)
        self.cfg = config
        self.level = 0                     # 当前待放的层号（0 开始）
        self.current_ws = None
        self.offset_pick = (0.0, 0.0)      # 抓取区定位扰动（仿真用）
        self.offset_place = (0.0, 0.0)     # 堆叠区定位扰动（仿真用）
        self.ws_path = []
        self.block_log = []
        self.tower = []                    # 每层释放点（工作台坐标）
        self.is_virtual = bool(getattr(arm, "is_virtual", False))
        self.sleep = (lambda s: None) if self.is_virtual else time.sleep

    # ---------- 坐标与基本动作 ----------
    def tf_ws2arm(self, wx, wy, wz):
        v = self.T_arm2ws @ np.array([wx, wy, wz, 1.0], dtype=np.float64)
        return float(v[0]), float(v[1]), float(v[2])

    def move_ws(self, wx, wy, wz, t=0.6, tag=""):
        ax, ay, az = self.tf_ws2arm(wx, wy, wz)
        ok = self.arm.set_tool_pose([ax, ay, az], T=t)
        if ok is False:
            raise SafetyError(
                "逆运动学无解: ws=({:.2f}, {:.2f}, {:.2f})".format(wx, wy, wz))
        self.current_ws = (float(wx), float(wy), float(wz))
        self.ws_path.append({"x": float(wx), "y": float(wy),
                             "z": float(wz), "tag": tag})

    def move_z(self, x, y, z0, z1, steps=6, t=0.5, tag=""):
        zs = np.linspace(float(z0), float(z1), int(steps) + 1)[1:]
        for z in zs:
            self.move_ws(x, y, float(z), t=t, tag=tag)

    def move_xy(self, x0, y0, x1, y1, z, steps=6, t=0.8, tag=""):
        xs = np.linspace(float(x0), float(x1), int(steps) + 1)[1:]
        ys = np.linspace(float(y0), float(y1), int(steps) + 1)[1:]
        for x, y in zip(xs, ys):
            self.move_ws(float(x), float(y), float(z), t=t, tag=tag)

    # ---------- 复用第 3 章的 Pick / Place ----------
    def pick_from(self, zone):
        """Approach → Pick → Lift：从抓取区取一块方块。"""
        if zone not in self.cfg.zones:
            raise SafetyError("未知抓取区: {}".format(zone))
        zx = self.cfg.zones[zone][0] + self.offset_pick[0]
        zy = self.cfg.zones[zone][1] + self.offset_pick[1]
        pz = self.cfg.zone_pick_z                # 14 mm：方块中心高度
        az = pz + self.cfg.approach_clearance    # 抓取上方点

        self.arm.gripper_open(angle=self.cfg.open_angle, T=0.5)
        self.move_ws(zx, zy, self.cfg.safe_z, t=1.0, tag="zone_safe")
        self.move_z(zx, zy, self.cfg.safe_z, az, steps=6, t=0.6, tag="zone_approach")
        self.move_z(zx, zy, az, pz, steps=self.cfg.pick_steps, t=0.45, tag="zone_descend")
        self.arm.gripper_close(T=0.5)
        self.sleep(self.cfg.settle_s)
        self.move_z(zx, zy, pz, az, steps=self.cfg.pick_steps, t=0.45, tag="zone_lift")
        self.move_z(zx, zy, az, self.cfg.safe_z, steps=6, t=0.6, tag="zone_retreat")

    def place_at_level(self, level):
        """Transfer → Place → Release → Retreat：把方块放到第 level 层。"""
        if level != self.level:
            raise SafetyError(
                "堆叠顺序错误：请求第{}层，当前应放第{}层".format(level + 1, self.level + 1))
        sx = self.cfg.stack_xy[0] + self.offset_place[0]
        sy = self.cfg.stack_xy[1] + self.offset_place[1]
        rz = self.cfg.stack_z(level)             # 目标中心高度（层号参数化）
        cz = self.cfg.approach_z(level)          # 放置上方点 = 目标点 + 60 mm

        # 安全检查：放置上方点必须高于当前塔顶，否则横移会撞塔
        if cz < self.cfg.tower_top_z(level) + 20.0:
            raise SafetyError("放置上方点过低，可能碰到塔顶")
        # 安全检查：横移必须在安全高度完成
        if self.cfg.safe_z <= cz:
            raise SafetyError("安全高度不足以跨越放置上方点")

        self.move_xy(self.current_ws[0], self.current_ws[1], sx, sy,
                     self.cfg.safe_z, steps=self.cfg.transfer_steps, t=0.8, tag="transfer")
        self.move_z(sx, sy, self.cfg.safe_z, cz, steps=6, t=0.6, tag="stack_approach")
        self.move_z(sx, sy, cz, rz, steps=self.cfg.place_steps, t=0.45, tag="stack_descend")
        self.arm.gripper_open(angle=self.cfg.open_angle, T=0.5)
        self.sleep(self.cfg.settle_s * 2.0)      # 释放后多等一拍，让方块落稳
        self.move_z(sx, sy, rz, cz, steps=self.cfg.place_steps, t=0.45, tag="stack_release")
        self.move_z(sx, sy, cz, self.cfg.safe_z, steps=self.cfg.retreat_steps, t=0.6, tag="stack_retreat")
        self.arm.gripper_close(T=0.5)            # 收拢夹爪再横移，避免剐蹭塔体

        self.tower.append((float(sx), float(sy), float(rz), int(level)))
        self.level += 1

    def run(self, round_idx=0):
        self.arm.home()
        self.arm.gripper_open(angle=self.cfg.open_angle, T=0.5)
        for order, (level, zone) in enumerate(self.cfg.plan(), start=1):
            try:
                self.pick_from(zone)
                self.place_at_level(level)
                status = "成功"
            except Exception as exc:
                status = "失败: {}".format(exc)
                self.block_log.append({
                    "round": round_idx, "order": order, "level": level + 1,
                    "zone": zone, "pick_z": round(self.cfg.zone_pick_z, 2),
                    "stack_z": round(self.cfg.stack_z(level), 2),
                    "approach_z": round(self.cfg.approach_z(level), 2),
                    "offset_pick_x": round(self.offset_pick[0], 3),
                    "offset_pick_y": round(self.offset_pick[1], 3),
                    "offset_place_x": round(self.offset_place[0], 3),
                    "offset_place_y": round(self.offset_place[1], 3),
                    "status": status,
                })
                break
            self.block_log.append({
                "round": round_idx, "order": order, "level": level + 1,
                "zone": zone, "pick_z": round(self.cfg.zone_pick_z, 2),
                "stack_z": round(self.cfg.stack_z(level), 2),
                "approach_z": round(self.cfg.approach_z(level), 2),
                "offset_pick_x": round(self.offset_pick[0], 3),
                "offset_pick_y": round(self.offset_pick[1], 3),
                "offset_place_x": round(self.offset_place[0], 3),
                "offset_place_y": round(self.offset_place[1], 3),
                "status": status,
            })
        self.arm.home()
        return self.block_log

    def release(self):
        try:
            self.arm.set_damping(1000)
        except Exception:
            pass

print("StackTask 定义完成")

## 6. 可达性与安全预检查

对每一层的 5 个关键点逐一做逆运动学检查：

1. **抓取上方点**：抓取区上方 `Z = 74 mm`；
2. **抓取点**：抓取区方块中心 `Z = 14 mm`；
3. **抬升点**：抓取区上升到安全高度 `Z = 200 mm`；
4. **放置上方点**：中央堆叠区 `Z = stack_z(level) + 60 mm`；
5. **放置点（释放点）**：中央堆叠区 `Z = stack_z(level)`。

只有 5 个点全部有合法解，才允许进入正式运行。第 4 层的放置上方点为 `98 + 1 + 60 = 159 mm`，仍然低于安全高度 200 mm，满足“先升到安全高度、再横移到目标上方”的顺序约束。

In [ ]:
kin_check = Arm5DoFKinematic()

def adjusted_arm_pose(wx, wy, wz):
    p = (T_ARM2WS @ np.array([wx, wy, wz, 1.0], dtype=np.float64))[:3]
    k, b = Z_ERROR_POLY
    r = math.hypot(p[0], p[1])
    p = p.copy()
    p[2] += k * r + b
    return p

def ik_solution_count(wx, wy, wz):
    p = adjusted_arm_pose(wx, wy, wz)
    pitch = PITCH_PANEL_A * math.hypot(p[0], p[1]) + PITCH_PANEL_B * p[2] + PITCH_PANEL_C
    sols = kin_check.inverse_kinematic(
        p[0], p[1], p[2], pitch, np.pi,
        last_joint_angle=np.radians([0, -90, 90, 0, 0]))
    return len(sols), math.degrees(pitch), p

rows = []
all_reachable = True
for level, zone in cfg.plan():
    zx, zy = cfg.zones[zone]
    sx, sy = cfg.stack_xy
    check_points = [
        ("抓取上方点", zx, zy, cfg.zone_pick_z + cfg.approach_clearance),
        ("抓取点", zx, zy, cfg.zone_pick_z),
        ("抬升点", zx, zy, cfg.safe_z),
        ("放置上方点", sx, sy, cfg.approach_z(level)),
        ("放置点", sx, sy, cfg.stack_z(level)),
    ]
    counts = []
    pitches = []
    for name, x, y, z in check_points:
        n, pitch_deg, _ = ik_solution_count(x, y, z)
        counts.append(n)
        pitches.append(pitch_deg)
        if n == 0:
            all_reachable = False
    rows.append((level, zone, counts, pitches))

html = ['<table border="1" cellspacing="0" cellpadding="5" style="border-collapse:collapse;font-size:13px">']
html.append('<tr style="background:#e8f1fb"><th>层号</th><th>抓取区</th>'
            '<th>抓取上方点</th><th>抓取点</th><th>抬升点</th>'
            '<th>放置上方点</th><th>放置点</th><th>俯仰角范围/°</th></tr>')
for level, zone, counts, pitches in rows:
    cells_txt = ''.join(
        '<td style="text-align:center;color:{}">{}</td>'.format(
            '#1a7f37' if n > 0 else '#c62828',
            '可达({}解)'.format(n) if n > 0 else '不可达')
        for n in counts)
    html.append(
        '<tr><td style="text-align:center">第{}层</td><td style="text-align:center">{}</td>{}</tr>'.format(
            level + 1, zone, cells_txt))
html.append('</table>')
display(HTML(''.join(html)))
print("全部关键点可达：", all_reachable)

## 7. 连续 2 轮堆叠与稳定性分析

### 7.1 运动学仿真（2 轮）

每轮给抓取区和堆叠区各加 ±1.0 mm 的随机定位扰动，检查在轻微标定/识别误差下，四层动作是否仍能求出合法逆解、顺序是否正确、退出路径是否安全。  
注意：这一步验证的是**运动学可达性、动作顺序、退出路径和安全高度**，不包含真机摩擦、夹持滑移和方块真实碰撞。

### 7.2 塔体稳定性模型

高层塔对 XY 偏差最敏感，本 Notebook 用一个可解释的简化模型做误差分析：

- 单层落块偏差 `e_i ~ N(0, σ²)`，σ 默认 1.0 mm（重复定位误差 + 释放滑移的合成）；
- **绝对定位模式**：每层都示教同一个中央点，`c_i = e_i`，各层偏差互不累加；
- **相对堆叠模式**：每层相对上一层放置，`c_i = c_{i-1} + e_i`，误差逐层累积；
- 稳定性判据（方块边长 28 mm，支撑半宽 14 mm）：
  - **层间错位** `|c_i − c_{i−1}| ≤ 12 mm`：保证每块仍坐在下面一块上（≤ 0.43 × 边长）；
  - **塔体重心投影** `|mean(c₁…c₄)| ≤ 7 mm`：保证整塔重心不越出底座（≤ 0.25 × 边长）；
  - **塔顶悬空** `|c₄| ≤ 14 mm`：塔顶方块不能越出底座半宽。
  三个条件同时满足才判为“塔体稳定”，否则记录最先失稳的层号。

再用蒙特卡洛扫描 σ = 0～8 mm，比较两种模式的稳定率：绝对定位时塔体重心偏移只有 σ/2，相对堆叠时相邻两层偏差相同、但整塔重心偏移会放大到约 1.37σ，并且塔顶偏差是各层误差之和——这正是“高层塔更容易倒”的数学原因。

In [ ]:
# ===================== 塔体稳定性模型 =====================
CUBE_HALF_MM = cfg.cube_size / 2.0     # 14 mm，单块支撑半宽
LAYER_TOL_MM = 12.0                    # 层间错位容差（≤ 0.43 × 边长，保证还坐在下层上）
COM_TOL_MM = 7.0                       # 塔体重心投影容差（≤ 0.25 × 边长，防倾覆）
TOP_TOL_MM = CUBE_HALF_MM              # 塔顶悬空容差 = 半宽
N_LAYER = len(cfg.pick_order)

def sample_layer_offsets(rng, sigma=1.0, mode="absolute", n=N_LAYER):
    """按单层定位误差 sigma 生成各层累积偏差，返回 n×2 数组。

    mode="absolute"：每层都放到同一个绝对点，c_i = e_i；
    mode="relative"：每层相对上一层放置，c_i = c_{i-1} + e_i（误差逐层累积）。
    """
    errs = rng.normal(0.0, sigma, size=(n, 2))
    out = []
    c = np.zeros(2)
    for k in range(n):
        c = c + errs[k] if mode == "relative" else errs[k]
        out.append(c.copy())
    return np.array(out)

def judge_tower(offsets):
    """返回 (是否稳定, 最大层间错位, 重心偏移, 塔顶偏差, 最先失稳层号)。"""
    prev = np.zeros(2)
    max_rel, fail_level = 0.0, None
    for i, c in enumerate(offsets):
        d = float(np.linalg.norm(c - prev))
        max_rel = max(max_rel, d)
        if d > LAYER_TOL_MM and fail_level is None:
            fail_level = i + 1
        prev = c
    com = float(np.linalg.norm(np.mean(offsets, axis=0)))
    top = float(np.linalg.norm(offsets[-1]))
    if com > COM_TOL_MM and fail_level is None:
        fail_level = N_LAYER
    if top > TOP_TOL_MM and fail_level is None:
        fail_level = N_LAYER
    stable = (max_rel <= LAYER_TOL_MM) and (com <= COM_TOL_MM) and (top <= TOP_TOL_MM)
    return stable, max_rel, com, top, (fail_level if not stable else None)

# ===================== 7.1 连续 2 轮堆叠仿真 =====================
SIM_TASKS, SIM_LOGS, ROUND_RESULTS = [], [], []

if RUN_MODE != "sim":
    print("当前 RUN_MODE = {}，跳过堆叠仿真。".format(RUN_MODE))
else:
    for r in range(2):
        rng = np.random.default_rng(2026 + r)
        dx1, dy1, dx2, dy2 = rng.uniform(-1.0, 1.0, 4)
        arm_v = VirtualArm(T_ARM2WS, Z_ERROR_POLY)
        task = StackTask(arm_v, T_ARM2WS, cfg)
        task.offset_pick = (float(dx1), float(dy1))
        task.offset_place = (float(dx2), float(dy2))
        logs = task.run(round_idx=r + 1)
        SIM_TASKS.append(task)
        SIM_LOGS.extend(logs)

        # 同一随机源做逐层偏差分析（绝对定位模式）
        offsets = sample_layer_offsets(rng, sigma=1.0, mode="absolute")
        stable, worst, com, top, fail_level = judge_tower(offsets)
        prev = np.zeros(2)
        layer_dev = []
        for i, c in enumerate(offsets):
            layer_dev.append({
                "level": i + 1,
                "zone": cfg.pick_order[i],
                "cmd_z": round(cfg.stack_z(i), 2),
                "dev_x": round(float(c[0]), 3),
                "dev_y": round(float(c[1]), 3),
                "dev_norm": round(float(np.linalg.norm(c)), 3),
                "rel_to_prev": round(float(np.linalg.norm(c - prev)), 3),
            })
            prev = c
        ok_blocks = sum(1 for it in logs if it["status"] == "成功")
        ROUND_RESULTS.append({
            "round": r + 1, "blocks": ok_blocks, "total": N_LAYER,
            "ik_fail": arm_v.ik_fail, "commands": arm_v.command_count,
            "offset_pick": (round(float(dx1), 3), round(float(dy1), 3)),
            "offset_place": (round(float(dx2), 3), round(float(dy2), 3)),
            "stable": stable, "max_rel": round(worst, 3),
            "com_offset": round(com, 3), "top_dev": round(top, 3),
            "fail_level": fail_level, "layers": layer_dev,
        })
        print("第{}轮：{}/{} 块成功，IK失败={}，指令数={}，层间最大错位={:.2f} mm，"
              "重心偏移={:.2f} mm，塔顶偏差={:.2f} mm，塔体稳定={}".format(
                  r + 1, ok_blocks, N_LAYER, arm_v.ik_fail, arm_v.command_count,
                  worst, com, top, "是" if stable else "否"))
        print("        抓取扰动={:+.2f}/{:+.2f} mm，放置扰动={:+.2f}/{:+.2f} mm".format(
            dx1, dy1, dx2, dy2))

# ===================== 7.2 误差敏感性扫描（蒙特卡洛） =====================
SIGMA_LIST = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0, 8.0]
N_TRIAL = 1000
SWEEP = {"absolute": [], "relative": []}
for mode in SWEEP:
    for sigma in SIGMA_LIST:
        rng = np.random.default_rng(7)
        ok = 0
        for _ in range(N_TRIAL):
            stable, _, _, _, _ = judge_tower(sample_layer_offsets(rng, sigma, mode))
            ok += int(stable)
        SWEEP[mode].append(ok / N_TRIAL)

print()
print("{:<10} {:>14} {:>14}".format("σ/mm", "绝对定位稳定率", "相对堆叠稳定率"))
for i, sigma in enumerate(SIGMA_LIST):
    print("{:<10.1f} {:>14.1f}% {:>14.1f}%".format(
        sigma, 100 * SWEEP["absolute"][i], 100 * SWEEP["relative"][i]))

# 两种模式各自的临界 σ（稳定率首次跌破 50%）
def critical_sigma(rates):
    """稳定率首次跌破 50% 的 sigma，用线性插值细化。"""
    for i in range(1, len(rates)):
        if rates[i] < 0.5 <= rates[i - 1]:
            s0, s1 = SIGMA_LIST[i - 1], SIGMA_LIST[i]
            r0, r1 = rates[i - 1], rates[i]
            return round(s0 + (0.5 - r0) * (s1 - s0) / (r1 - r0), 2)
    return None

crit_abs = critical_sigma(SWEEP["absolute"])
crit_rel = critical_sigma(SWEEP["relative"])
print()
print("绝对定位模式临界 σ（稳定率 50%）≈", crit_abs, "mm")
print("相对堆叠模式临界 σ（稳定率 50%）≈", crit_rel, "mm")
print("说明：相邻两层的相对错位在两种模式下都是 |e_i|，"
      "但绝对定位时塔体重心偏移 ≈ σ/2，相对堆叠时会放大到 ≈ 1.37σ，"
      "塔顶偏差更会累加成各层误差之和，所以相对堆叠在高 σ 下先失稳。")

## 8. 结果汇总与测试记录

生成以下文件：

- `results/ch4_stack_pose_table.csv`：stack_pose 参数表（每层的抓取区、抓取点、目标高度、放置上方点、塔顶高度、退出方向）；
- `results/ch4_stability_test.csv`：连续 2 轮的逐层落块偏差与稳定性记录；
- `results/ch4_sensitivity.csv`：误差敏感性扫描结果（σ  vs  稳定率）；
- `results/ch4_calibration.csv`：九点实测值与残差；
- `results/ch4_real_test_template.csv`：真机测试记录空表，现场填写。

In [ ]:
# ---------- 轮次汇总 ----------
summary = ROUND_RESULTS

total_ok = sum(x["blocks"] for x in summary)
total_n = sum(x["total"] for x in summary)
print("连续 {} 轮堆叠：{}/{} 块次成功，塔体稳定 {}/{} 轮".format(
    len(summary), total_ok, total_n,
    sum(1 for x in summary if x["stable"]), len(summary)))

html = ['<table border="1" cellspacing="0" cellpadding="5" style="border-collapse:collapse;font-size:13px">']
html.append('<tr style="background:#e8f1fb"><th>轮次</th><th>成功块次</th><th>IK失败</th>'
            '<th>指令数</th><th>层间最大错位/mm</th><th>重心偏移/mm</th>'
            '<th>塔顶偏差/mm</th><th>塔体稳定</th><th>失稳层号</th></tr>')
for s in summary:
    html.append('<tr><td style="text-align:center">第{}轮</td>'
                '<td style="text-align:center">{}/{}</td>'
                '<td style="text-align:center">{}</td>'
                '<td style="text-align:center">{}</td>'
                '<td style="text-align:center">{:.2f}</td>'
                '<td style="text-align:center">{:.2f}</td>'
                '<td style="text-align:center">{:.2f}</td>'
                '<td style="text-align:center;color:{}">{}</td>'
                '<td style="text-align:center">{}</td></tr>'.format(
                    s["round"], s["blocks"], s["total"], s["ik_fail"], s["commands"],
                    s["max_rel"], s["com_offset"], s["top_dev"],
                    "#1a7f37" if s["stable"] else "#c62828",
                    "稳定" if s["stable"] else "失稳",
                    "—" if s["fail_level"] is None else "第{}层".format(s["fail_level"])))
html.append('</table>')
display(HTML(''.join(html)))

# ---------- 保存 CSV ----------
pose_csv = os.path.join(RESULT_DIR, "ch4_stack_pose_table.csv")
with open(pose_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["stack_level", "zone", "zone_x", "zone_y", "pick_z",
                     "zone_approach_z", "stack_x", "stack_y", "target_z",
                     "stack_approach_z", "tower_top_z", "safe_z", "exit_direction"])
    for level, zone in cfg.plan():
        zx, zy = cfg.zones[zone]
        writer.writerow([level + 1, zone, zx, zy, cfg.zone_pick_z,
                         cfg.zone_pick_z + cfg.approach_clearance,
                         cfg.stack_xy[0], cfg.stack_xy[1], cfg.stack_z(level),
                         cfg.approach_z(level), cfg.tower_top_z(level),
                         cfg.safe_z, "纯垂直上升 +Z"])

stab_csv = os.path.join(RESULT_DIR, "ch4_stability_test.csv")
with open(stab_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["round", "level", "zone", "target_z", "dev_x", "dev_y",
                     "dev_norm", "rel_to_prev", "round_stable",
                     "max_rel_mm", "com_offset_mm", "top_dev_mm", "fail_level"])
    for res in summary:
        for L in res["layers"]:
            writer.writerow([res["round"], L["level"], L["zone"], L["cmd_z"],
                             L["dev_x"], L["dev_y"], L["dev_norm"], L["rel_to_prev"],
                             res["stable"], res["max_rel"], res["com_offset"],
                             res["top_dev"], res["fail_level"] or ""])

sens_csv = os.path.join(RESULT_DIR, "ch4_sensitivity.csv")
with open(sens_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["sigma_mm", "absolute_rate", "relative_rate"])
    for i, sigma in enumerate(SIGMA_LIST):
        writer.writerow([sigma, SWEEP["absolute"][i], SWEEP["relative"][i]])

calib_csv = os.path.join(RESULT_DIR, "ch4_calibration.csv")
with open(calib_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["point", "arm_x", "arm_y", "arm_z", "residual_mm"])
    for i in range(9):
        writer.writerow(["P{}".format(i), *ARM_9P_MEASURED[i], RESIDUAL_NORM[i]])

real_csv = os.path.join(RESULT_DIR, "ch4_real_test_template.csv")
with open(real_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["round", "level", "zone", "target_z", "success",
                     "xy_deviation_mm", "tower_stable", "failure_reason", "note"])
    for r in range(1, 3):
        for level, zone in cfg.plan():
            writer.writerow([r, level + 1, zone, cfg.stack_z(level), "", "", "", "", ""])

print("已保存：")
for p in [pose_csv, stab_csv, sens_csv, calib_csv, real_csv]:
    print(" -", os.path.abspath(p))

## 9. 流程图与工作台布局图

图 1 是单块方块的堆叠动作流程（与第 3 章共用同一套六阶段，但放置点随层号变化）；图 2 是工作台布局：A/B/C/D 四个抓取区、中央堆叠区、九点标定板以及抓取顺序。

In [ ]:
# ---------- 图 1：堆叠动作流程图 ----------
def draw_flowchart(path):
    fig, ax = plt.subplots(figsize=(9.8, 4.8))
    ax.set_xlim(0, 9.8)
    ax.set_ylim(0, 4.8)
    ax.axis("off")
    steps = [
        ("Approach\n移到抓取区上方", "Z = 74 mm，夹爪张开"),
        ("Pick\n下降到抓取点", "Z = 14 mm，分段下降 5 步"),
        ("Lift\n夹紧后抬升", "回到 Z = 200 mm"),
        ("Transfer\n安全高度横移", "Z = 200 mm 移到中央堆叠区"),
        ("Place\n按层号下降释放", "Z = stack_z(level)，开爪释放"),
        ("Retreat\n纯垂直退出", "垂直上升，不拖动方块"),
    ]
    colors = ["#dbeafe", "#dbeafe", "#dcfce7", "#fef3c7", "#fee2e2", "#ede9fe"]
    w, h = 2.55, 1.30
    xs = [0.35, 3.45, 6.55]
    y_top, y_bot = 2.65, 0.75
    positions = [(xs[0], y_top), (xs[1], y_top), (xs[2], y_top),
                 (xs[2], y_bot), (xs[1], y_bot), (xs[0], y_bot)]
    for i, ((title, sub), (x, y)) in enumerate(zip(steps, positions), start=1):
        box = FancyBboxPatch((x, y), w, h,
                             boxstyle="round,pad=0.03,rounding_size=0.10",
                             facecolor=colors[i - 1], edgecolor="#334155", linewidth=1.3)
        ax.add_patch(box)
        ax.text(x + 0.28, y + h - 0.24, "{}".format(i), ha="center", va="center",
                fontsize=9, color="white", fontweight="bold",
                bbox=dict(boxstyle="circle,pad=0.24", fc="#334155", ec="none"))
        ax.text(x + w / 2, y + h * 0.63, title, ha="center", va="center",
                fontsize=11.0, fontweight="bold")
        ax.text(x + w / 2, y + h * 0.22, sub, ha="center", va="center", fontsize=8.6)

    def arrow(p0, p1):
        ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=15,
                                     linewidth=1.4, color="#475569"))
    arrow((xs[0] + w, y_top + h / 2), (xs[1], y_top + h / 2))
    arrow((xs[1] + w, y_top + h / 2), (xs[2], y_top + h / 2))
    arrow((xs[2] + w / 2, y_top), (xs[2] + w / 2, y_bot + h))
    arrow((xs[2], y_bot + h / 2), (xs[1] + w, y_bot + h / 2))
    arrow((xs[1], y_bot + h / 2), (xs[0] + w, y_bot + h / 2))

    ax.text(4.9, 4.45, "单块方块堆叠动作流程（第 3 章 Pick & Place 的反向复用）", ha="center",
            fontsize=14, fontweight="bold")
    ax.text(4.9, 0.30, "安全约束：抓取/放置上方点 = 目标点 + 60 mm；横移只在 Z = 200 mm；"
                       "释放后纯垂直退出；抓放垂直段分 5～8 步。",
            ha="center", fontsize=9.0, color="#334155")
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

# ---------- 图 2：工作台布局 ----------
def draw_workspace(path):
    fig, ax = plt.subplots(figsize=(7.8, 7.2))
    ax.add_patch(Rectangle((-X0, -Y0), 2 * X0, 2 * Y0, fill=False,
                           edgecolor="#64748b", linewidth=1.2, linestyle="--"))
    p_points = {
        "P0": (X0, Y0), "P1": (X0, 0), "P2": (X0, -Y0),
        "P3": (0, Y0), "P4": (0, 0), "P5": (0, -Y0),
        "P6": (-X0, Y0), "P7": (-X0, 0), "P8": (-X0, -Y0),
    }
    label_offsets = {
        "P0": (-6.0, -5.0), "P1": (-6.0, 0.0), "P2": (-6.0, 5.0),
        "P3": (0.0, 5.0), "P4": (2.0, 2.0), "P5": (0.0, -7.0),
        "P6": (6.0, -5.0), "P7": (6.0, 0.0), "P8": (6.0, 5.0),
    }
    for name, (x, y) in p_points.items():
        ax.plot(x, y, "o", color="#64748b", markersize=4)
        dx, dy = label_offsets[name]
        ax.text(x + dx, y + dy, name, fontsize=8, color="#475569")

    # 中央 4 层塔（俯视用同心的 4 个方块示意）
    stack_colors = ["#fca5a5", "#fdba74", "#fde68a", "#bbf7d0"]
    for i in range(4):
        off = i * 1.6
        ax.add_patch(Rectangle((-14 + off, -14 + off), 28, 28,
                               facecolor=stack_colors[i], edgecolor="#1f2937",
                               linewidth=1.0, alpha=0.92))
    ax.text(0, -24, "中央堆叠区\n第1层→第4层", ha="center", va="top", fontsize=9.5)

    order_label = {z: str(i + 1) for i, z in enumerate(cfg.pick_order)}
    for name, (x, y) in cfg.zones.items():
        ax.add_patch(Rectangle((x - 15, y - 15), 30, 30,
                               facecolor="#eff6ff", edgecolor="#2563eb",
                               linewidth=1.4, linestyle="--", alpha=0.95))
        ax.text(x, y + 1.5, name, ha="center", va="center",
                fontsize=15, fontweight="bold", color="#1d4ed8")
        ax.text(x, y - 8.5, "第{}层".format(order_label[name]), ha="center",
                va="center", fontsize=8.5, color="#1e3a8a")
        ax.add_patch(FancyArrowPatch((x, y), (0, 0), arrowstyle="-|>",
                                     mutation_scale=12, linewidth=1.0,
                                     color="#94a3b8", linestyle=":"))
    ax.text(0, -58, "抓取顺序 A → B → C → D", ha="center", fontsize=10, color="#374151")

    ax.set_xlim(-88, 88)
    ax.set_ylim(-104, 104)
    ax.set_aspect("equal")
    ax.set_xlabel("工作台 X / mm")
    ax.set_ylabel("工作台 Y / mm")
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_title("工作台坐标系、A/B/C/D 抓取区与中央堆叠区", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

flow_path = os.path.join(IMAGE_DIR, "ch4_flowchart.png")
workspace_path = os.path.join(IMAGE_DIR, "ch4_workspace.png")
draw_flowchart(flow_path)
draw_workspace(workspace_path)
print("已保存：", flow_path)
print("已保存：", workspace_path)

## 10. 标定残差、层高、轨迹与稳定性结果图

图 3 九点标定残差与 Z 误差补偿；图 4 四层堆叠的目标高度、释放间隙与退出路径；图 5 第一轮仿真轨迹；图 6 第一轮各层落块偏差与塔体倾斜；图 7 连续 2 轮堆叠结果；图 8 误差敏感性扫描（绝对定位 vs 相对堆叠）。

In [ ]:
# ---------- 图 3：九点标定残差 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
ax.quiver(ARM_9P_FLAT[:, 0], ARM_9P_FLAT[:, 1],
          RESIDUAL[:, 0], RESIDUAL[:, 1],
          angles="xy", scale_units="xy", scale=1.0,
          color="#dc2626", width=0.004)
ax.scatter(ARM_9P_FLAT[:, 0], ARM_9P_FLAT[:, 1], s=26, c="#1f2937", zorder=3)
for i in range(9):
    ax.text(ARM_9P_FLAT[i, 0] + 1.2, ARM_9P_FLAT[i, 1] + 1.2, "P{}".format(i), fontsize=8)
ax.set_aspect("equal")
ax.set_xlim(ARM_9P_FLAT[:, 0].min() - 14, ARM_9P_FLAT[:, 0].max() + 14)
ax.set_ylim(ARM_9P_FLAT[:, 1].min() - 14, ARM_9P_FLAT[:, 1].max() + 14)
ax.set_xlabel("机械臂 X / mm")
ax.set_ylabel("机械臂 Y / mm")
ax.set_title("九点 XY 残差（max {:.2f} mm）".format(RESIDUAL_NORM.max()), fontsize=11)
ax.grid(True, linestyle=":", alpha=0.4)

ax = axes[1]
z_err = ARM_9P_MEASURED[:, 2] - Z_MEAN
ax.scatter(RADII, z_err, s=38, c="#1d4ed8", label="实测 z 误差")
rr = np.linspace(RADII.min(), RADII.max(), 100)
ax.plot(rr, np.polyval(Z_ERROR_POLY, rr), "r-", linewidth=1.6, label="拟合 z_error(r)")
ax.set_xlabel("半径 r / mm")
ax.set_ylabel("z - z_mean / mm")
ax.set_title("Z 误差随半径的线性补偿", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.4)
ax.legend(fontsize=8.5)

fig.tight_layout()
calib_path = os.path.join(IMAGE_DIR, "ch4_calibration.png")
fig.savefig(calib_path, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

# ---------- 图 4：四层堆叠目标高度与退出路径 ----------
def draw_stack_profile(path):
    fig, ax = plt.subplots(figsize=(9.0, 6.6))
    h, gap = cfg.cube_size, cfg.release_clearance
    ax.axhline(0, color="#444444", linewidth=2.0)
    ax.text(-34, -6, "工作台面 Z=0", fontsize=9, color="#444444", ha="left", va="top")
    colors = ["#fca5a5", "#fdba74", "#fde68a", "#bbf7d0"]
    for i in range(4):
        ax.add_patch(Rectangle((-16, i * h), 32, h, facecolor=colors[i],
                               edgecolor="#1f2937", linewidth=1.0, alpha=0.95))
        ax.text(0, i * h + h / 2, "第{}层 {}/{}\n中心 {:.0f} mm".format(
            i + 1, cfg.pick_order[i], cfg.zones[cfg.pick_order[i]],
            (i + 0.5) * h), ha="center", va="center", fontsize=8.2)
    for i in range(4):
        z = cfg.stack_z(i)
        ax.plot([-30, -16], [z, z], color="#dc2626", linewidth=1.4)
        ax.text(-31, z, "释放点 {:.0f}".format(z), ha="right", va="center",
                fontsize=8.5, color="#991b1b")
        za = cfg.approach_z(i)
        ax.plot([16, 30], [za, za], color="#2563eb", linewidth=1.4)
        ax.text(31, za, "放置上方点 {:.0f}".format(za), ha="left", va="center",
                fontsize=8.5, color="#1d4ed8")
    ax.annotate("", xy=(50, cfg.safe_z), xytext=(50, 4 * h),
                arrowprops=dict(arrowstyle="-|>", color="#0f766e", lw=1.6))
    ax.text(51, (cfg.safe_z + 4 * h) / 2, "退出路径：纯垂直 +Z", fontsize=9,
            color="#0f766e", rotation=90, va="center")
    ax.axhline(cfg.safe_z, color="#0f766e", linestyle="--", linewidth=1.4)
    ax.text(-34, cfg.safe_z + 3, "安全高度 Z={:.0f} mm".format(cfg.safe_z),
            ha="left", va="bottom", fontsize=9.5, color="#0f766e")
    ax.text(0, 4 * h + 60, "stack_z(level) = 方块高度 × (level + 0.5) + 释放间隙({:.1f} mm)".format(gap),
            ha="center", va="bottom", fontsize=10, color="#374151",
            bbox=dict(boxstyle="round,pad=0.35", fc="#f8fafc", ec="#cbd5e1"))
    ax.set_xlim(-46, 66)
    ax.set_ylim(-14, cfg.safe_z + 26)
    ax.set_xlabel("示意宽度 / mm")
    ax.set_ylabel("工作台 Z / mm")
    ax.set_title("四层堆叠目标高度、放置上方点与退出路径（h={:.0f} mm）".format(h),
                 fontsize=12, fontweight="bold")
    ax.grid(True, axis="y", linestyle=":", alpha=0.4)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

profile_path = os.path.join(IMAGE_DIR, "ch4_stack_profile.png")
draw_stack_profile(profile_path)

# ---------- 图 5：仿真轨迹 ----------
if SIM_TASKS:
    task0 = SIM_TASKS[0]
    xs = [p["x"] for p in task0.ws_path]
    ys = [p["y"] for p in task0.ws_path]
    zs = [p["z"] for p in task0.ws_path]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.0))
    ax = axes[0]
    ax.plot(xs, ys, "-", color="#64748b", linewidth=1.0, alpha=0.8)
    ax.scatter(xs[::5], ys[::5], s=8, c="#94a3b8")
    for name, (x, y) in cfg.zones.items():
        ax.scatter([x], [y], s=110, marker="^", c="#3b82f6", edgecolor="#1e3a8a",
                   label="抓取区" if name == cfg.pick_order[0] else None)
        ax.text(x + 2, y - 8, "{}（第{}层）".format(name, cfg.pick_order.index(name) + 1),
                fontsize=9, color="#1e3a8a")
    ax.scatter([0], [0], s=140, marker="s", c="#f59e0b", edgecolor="#7c2d12",
               label="中央堆叠区")
    ax.set_aspect("equal")
    ax.set_xlabel("工作台 X / mm")
    ax.set_ylabel("工作台 Y / mm")
    ax.set_title("第一轮仿真轨迹（俯视）", fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend(fontsize=8.5)

    ax = axes[1]
    idx = np.arange(len(zs))
    ax.plot(idx, zs, "-", color="#0f766e", linewidth=1.0, alpha=0.8)
    for level in range(4):
        z = cfg.stack_z(level)
        ax.axhline(z, color="#f59e0b", linestyle="--", linewidth=0.8, alpha=0.7)
        ax.text(1, z + 2, "第{}层释放 {:.0f} mm".format(level + 1, z),
                fontsize=8, color="#92400e")
    ax.axhline(cfg.safe_z, color="#dc2626", linestyle=":", linewidth=1.0)
    ax.text(1, cfg.safe_z + 2, "安全高度 200 mm", fontsize=8, color="#991b1b")
    ax.set_xlabel("轨迹点序号")
    ax.set_ylabel("工作台 Z / mm")
    ax.set_title("第一轮仿真轨迹（侧视）", fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.4)
    fig.tight_layout()
    traj_path = os.path.join(IMAGE_DIR, "ch4_trajectory.png")
    fig.savefig(traj_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()
else:
    traj_path = None

# ---------- 图 6：各层落块偏差与塔体倾斜 ----------
if ROUND_RESULTS:
    res0 = ROUND_RESULTS[0]
    layers = res0["layers"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.0))
    ax = axes[0]
    ax.axhline(0, color="#94a3b8", linestyle=":")
    ax.axvline(0, color="#94a3b8", linestyle=":")
    ax.add_patch(Rectangle((-CUBE_HALF_MM, -CUBE_HALF_MM), 2 * CUBE_HALF_MM, 2 * CUBE_HALF_MM,
                           fill=False, edgecolor="#1f2937", linewidth=1.2))
    ax.add_patch(Rectangle((-LAYER_TOL_MM, -LAYER_TOL_MM), 2 * LAYER_TOL_MM, 2 * LAYER_TOL_MM,
                           fill=False, edgecolor="#dc2626", linewidth=1.0, linestyle="--"))
    cols = ["#fca5a5", "#fdba74", "#fde68a", "#bbf7d0"]
    for i, L in enumerate(layers):
        ax.scatter([L["dev_x"]], [L["dev_y"]], s=120, marker="o", c=cols[i],
                   edgecolor="#1f2937", zorder=3, label="第{}层 {}".format(L["level"], L["zone"]))
        ax.annotate("", xy=(L["dev_x"], L["dev_y"]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="#6b7280", lw=1.0))
    ax.set_aspect("equal")
    ax.set_xlim(-16, 16)
    ax.set_ylim(-16, 16)
    ax.set_xlabel("X 偏差 / mm")
    ax.set_ylabel("Y 偏差 / mm")
    ax.set_title("第一轮各层落块 XY 偏差（虚线为 {} mm 层间错位容差）".format(LAYER_TOL_MM), fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend(fontsize=8.5, loc="upper right")
    axin = ax.inset_axes([0.07, 0.07, 0.40, 0.40])
    axin.set_xlim(-3, 3)
    axin.set_ylim(-3, 3)
    axin.axhline(0, color="#cbd5e1", linestyle=":")
    axin.axvline(0, color="#cbd5e1", linestyle=":")
    for i, L in enumerate(layers):
        axin.scatter([L["dev_x"]], [L["dev_y"]], s=70, marker="o", c=cols[i],
                     edgecolor="#1f2937", zorder=3)
    axin.set_title("局部放大 ±3 mm", fontsize=8)
    axin.tick_params(labelsize=7)
    axin.grid(True, linestyle=":", alpha=0.4)

    ax = axes[1]
    h = cfg.cube_size
    for i, L in enumerate(layers):
        dx = L["dev_x"] * 3.0   # 放大 3 倍便于观察
        ax.add_patch(Rectangle((-14 + dx, i * h), 28, h, facecolor=cols[i],
                               edgecolor="#1f2937", linewidth=1.0, alpha=0.92))
    ax.axhline(0, color="#444444", linewidth=2.0)
    ax.annotate("", xy=(layers[-1]["dev_x"] * 3.0, 4 * h + 16), xytext=(0, 4 * h + 16),
                arrowprops=dict(arrowstyle="-|>", color="#dc2626", lw=1.2))
    ax.text(0, 4 * h + 22, "塔顶偏差 {:.2f} mm（放大 3 倍绘制）".format(res0["top_dev"]),
            ha="center", fontsize=9.5, color="#991b1b")
    ax.set_xlim(-30, 30)
    ax.set_ylim(-8, 4 * h + 34)
    ax.set_aspect("equal")
    ax.set_xlabel("示意宽度 / mm")
    ax.set_ylabel("工作台 Z / mm")
    ax.set_title("第一轮塔体示意（层间错位 {:.2f} mm，重心偏移 {:.2f} mm）".format(res0["max_rel"], res0["com_offset"]), fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.4)
    fig.tight_layout()
    st_path = os.path.join(IMAGE_DIR, "ch4_stability.png")
    fig.savefig(st_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()
else:
    st_path = None

# ---------- 图 7：连续 2 轮堆叠结果 ----------
fig, ax = plt.subplots(figsize=(6.8, 4.2))
rounds = [s["round"] for s in summary]
success = [s["blocks"] for s in summary]
bars = ax.bar(["第{}轮".format(r) for r in rounds], success, color="#2563eb", width=0.55)
ax.set_ylim(0, 4.8)
ax.set_ylabel("成功块次 / 4")
ax.set_xlabel("测试轮次")
ax.set_title("连续 2 轮四方块自动堆叠结果（8/8 块次）", fontsize=11)
for b, v, s in zip(bars, success, summary):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.08, "{}/4".format(v),
            ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.text(b.get_x() + b.get_width() / 2, v / 2,
            "塔体{}".format("稳定" if s["stable"] else "失稳"),
            ha="center", va="center", fontsize=9.5, color="white", fontweight="bold")
ax.text(1.5, 0.35, "验收建议：机械臂退出后 4 层塔保持稳定，连续完成 2 轮",
        fontsize=8.5, color="#991b1b", ha="right")
ax.grid(axis="y", linestyle=":", alpha=0.4)
fig.tight_layout()
result_path = os.path.join(IMAGE_DIR, "ch4_test_result.png")
fig.savefig(result_path, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

# ---------- 图 8：误差敏感性扫描 ----------
fig, ax = plt.subplots(figsize=(7.6, 4.6))
ax.plot(SIGMA_LIST, [100 * v for v in SWEEP["absolute"]], "o-", color="#2563eb",
        linewidth=1.6, label="绝对定位（每层示教中央点）")
ax.plot(SIGMA_LIST, [100 * v for v in SWEEP["relative"]], "s--", color="#dc2626",
        linewidth=1.6, label="相对堆叠（每层相对上一层，误差累积）")
ax.axhline(50, color="#94a3b8", linestyle=":", linewidth=1.0)
ax.text(SIGMA_LIST[-1], 52, "稳定率 50%", fontsize=8.5, color="#64748b", ha="right")
ax.set_xlabel("单层定位误差 σ / mm")
ax.set_ylabel("4 层塔稳定率 / %")
ax.set_title("不同堆叠策略的误差敏感性（每点 {} 次蒙特卡洛）".format(N_TRIAL), fontsize=11)
if crit_abs is not None:
    ax.axvline(crit_abs, color="#2563eb", linestyle=":", linewidth=1.2)
    ax.text(crit_abs, 8, "绝对定位临界 σ≈{} mm".format(crit_abs), fontsize=8.5, color="#1d4ed8", rotation=90, va="bottom")
if crit_rel is not None:
    ax.axvline(crit_rel, color="#dc2626", linestyle=":", linewidth=1.2)
    ax.text(crit_rel, 8, "相对堆叠临界 σ≈{} mm".format(crit_rel), fontsize=8.5, color="#b91c1c", rotation=90, va="bottom")
ax.set_ylim(-3, 105)
ax.grid(True, linestyle=":", alpha=0.4)
ax.legend(fontsize=9)
fig.tight_layout()
sens_path = os.path.join(IMAGE_DIR, "ch4_sensitivity.png")
fig.savefig(sens_path, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

print("已保存：", calib_path)
print("已保存：", profile_path)
if traj_path:
    print("已保存：", traj_path)
if st_path:
    print("已保存：", st_path)
print("已保存：", result_path)
print("已保存：", sens_path)

## 11. 真机运行

真机运行前：

1. 把 `RUN_MODE` 改为 `"real"`；
2. 把 `REAL_CONFIRM` 改为 `True`；
3. 确认工作台无人、机械臂周围无杂物、急停可用；
4. 先把 A/B/C/D 四个方块摆正、彼此不接触，中央堆叠区清空；
5. 建议先只跑 `REAL_ROUNDS = 1`，重点观察第 3、4 层的落块偏差与退出时是否拖到塔体，再增加轮次。

> Windows 调试时默认串口不是 `/dev/ttyTHS1`，可改成 `Arm5DoFUServo(device="COM6")`。  
> 真机结果请填入 `results/ch4_real_test_template.csv`（round / level / zone / target_z / success / xy_deviation_mm / tower_stable / failure_reason / note）。

In [ ]:
def sync_calibration_to_config():
    """把本次九点标定结果同步到 config/，保证真机 SDK 使用同一组参数。"""
    os.makedirs("config", exist_ok=True)
    np.savetxt("config/arm9points.txt", ARM_9P_MEASURED, fmt="%.3f", delimiter=",")
    np.savetxt("config/z_error_polyfit.txt", Z_ERROR_POLY, fmt="%.4f", delimiter=",")
    np.savetxt("config/T_arm2ws.txt", T_ARM2WS, fmt="%.3f", delimiter=",")
    print("已同步九点标定参数到 config/")

def run_real_once(device=None):
    sync_calibration_to_config()
    from arm5dof_uservo import Arm5DoFUServo
    arm = Arm5DoFUServo(device=device) if device else Arm5DoFUServo()
    task = StackTask(arm, T_ARM2WS, cfg)
    logs = task.run(round_idx=99)
    task.release()
    return task, logs

if RUN_MODE != "real":
    print("当前为 sim 模式，真机单元未执行。")
elif not REAL_CONFIRM:
    print("已切换到 real，但 REAL_CONFIRM=False；确认安全后改为 True 再运行。")
else:
    for r in range(REAL_ROUNDS):
        task, logs = run_real_once()
        ok = sum(1 for x in logs if x["status"] == "成功")
        print("真机第{}轮：{}/4 成功".format(r + 1, ok))
        print("塔体：", [tuple(round(v, 2) for v in t[:3]) for t in task.tower])
        for item in logs:
            print("  ", item)

## 12. 结论与课后作业

### 12.1 本实验完成内容

- 复用 `08.四个堆叠方块归位.ipynb` 的九点标定、`T_arm2ws` 变换、逆运动学与 Pick & Place 六阶段框架，把“中央抓取 → 四区归位”反向改造成“四区抓取 → 中央逐层堆叠”；
- 用 `stack_z(level) = 方块高度 × (level + 0.5) + 释放间隙` 把四层目标高度参数化，**只有一套抓放逻辑**，四层塔由循环自动生成；
- 抓取顺序 A → B → C → D 对应自下而上第 1～4 层，每层放置前都做“放置上方点高于塔顶”“横移只在安全高度”的安全检查；
- 释放后采用**纯垂直退出**，并先收拢夹爪再横移，避免退出时拖倒新放的方块；
- 完成连续 2 轮堆叠仿真（8/8 块次、2/2 轮塔体稳定），并生成 stack_pose 参数表、稳定性测试记录与误差敏感性曲线；
- 用蒙特卡洛对比两种堆叠策略：绝对定位时塔体重心偏移只有单层误差的 σ/2，相对堆叠时相邻两层偏差相同、但整塔重心偏移放大到约 1.37σ、塔顶偏差累加为各层误差之和，所以相对堆叠在高 σ 下先失稳（σ 扫描到 8 mm 时，绝对定位临界 σ≈4.9 mm，相对堆叠只有≈4.1 mm；σ=3 mm 时两者稳定率分别为 95.3% 与 76.4%）——这正是高层塔“越堆越歪”的原因。

### 12.2 课后作业一：由 stack_level 自动计算目标高度（伪代码）

```text
输入: cube_height h           # 方块高度，28 mm
输入: release_clearance g     # 释放间隙，1 mm
输入: stack_level L           # 层号，自下而上从 0 开始

函数 stack_z(L):
    返回 h * (L + 0.5) + g     # 第 L 层方块中心的目标高度

函数 place_block(zone, L):
    抓取点   = zones[zone]      # 四区抓取，Z = h / 2
    放置点   = (0, 0, stack_z(L))
    上方点   = 放置点.Z + 60 mm
    依次执行 Approach → Pick → Lift → Transfer → Place → Retreat
    # 整个流程只依赖参数 L，不再复制四套代码
```

### 12.3 课后作业二：已经用公式算高度，为什么还需要标定？

1. **公式只给出“工作台坐标”下的高度**：`stack_z` 是理想几何关系，机械臂真正执行的是机械臂坐标系下的位姿，必须用 `T_arm2ws` 把 (x, y, z) 变换过去，而 `T_arm2ws` 只能由九点标定得到。
2. **Z 误差随半径变化**：实测九点 Z 值并不落在同一个平面上，本实验用 `z_error(r) = -0.0707 r + 13.8623` 做补偿。不标定就不知道这条曲线，抓取点和释放点都会整体偏高或偏低，最低层容易撞桌、高层容易“悬空释放”。
3. **公式假设方块高度严格等于 28 mm**：实际方块有加工/装配误差，真实塔高会有几毫米偏差，只有标定（或逐层实测）才能发现并补偿。
4. **公式不含机械误差**：舵机回差、连杆间隙、末端俯仰角误差、夹爪释放时的滑移都不在公式里，这些必须靠标定与实测数据修正。
5. **提高高层堆叠稳定性的措施**：① 每层都用绝对定位坐标而不是相对上一层，避免误差累积；② 释放点加 1 mm 释放间隙，释放后延时 0.5 s 再退出；③ 释放后纯垂直退出并先收拢夹爪；④ 第 3、4 层适当降低下降速度；⑤ 把单层定位误差控制在 σ ≤ 1 mm，必要时在第 3、4 层做一次落点复测与微调。

> 说明：本 Notebook 的 2 轮 8/8 块次与塔体稳定判定是**离线运动学与误差模型仿真结果**，不等价于真机成功率。真机测试请使用 `results/ch4_real_test_template.csv` 记录每层的成功/失败、XY 偏差与塔体状态。